In [1]:
%%capture
pip install gradio grad-cam

In [9]:
import gradio as gr
import torch
import torch.nn.functional as F
from facenet_pytorch import MTCNN, InceptionResnetV1
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [10]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Running on device: {DEVICE.upper()}')

Running on device: CUDA:0


In [11]:
mtcnn = MTCNN(
    select_largest=False,
    post_process=False,
    device=DEVICE
).to(DEVICE).eval()

model = InceptionResnetV1(
    pretrained="vggface2",
    classify=True,
    num_classes=1,
    device=DEVICE
)

use_checkpoint = True

if use_checkpoint:
    checkpoint = torch.load('./models/resnetinceptionv1_epoch_32.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(torch.load('./models/resnetinceptionv1_final.pth'))
model.to(DEVICE)
model.eval()
print("MTCNN & Classfier models loaded")

MTCNN & Classfier models loaded


In [12]:
EXAMPLES_FOLDER = 'examples'
examples_names = os.listdir(EXAMPLES_FOLDER)
examples = []
for example_name in examples_names:
    example_path = os.path.join(EXAMPLES_FOLDER, example_name)
    label = example_name.split('_')[0]
    example = {
        'path': example_path,
        'label': label
    }
    examples.append(example)
np.random.shuffle(examples) 

In [13]:
import cv2

In [14]:
example = examples[22]
example_img = example['path']
print(example['label'])
input_image = Image.open(example_img).convert('RGB')
print(input_image.mode)  # should say 'RGB'

fake
RGB


In [15]:
input_tensor = torch.from_numpy(np.array(input_image))
face = mtcnn(input_tensor)

if face is None:
    raise Exception('No face detected')

face = face.unsqueeze(0) 
face = F.interpolate(face, size=(256, 256), mode='bilinear', align_corners=False)

face_to_plot = face.squeeze(0).permute(1, 2, 0).cpu().detach().int().numpy()

plt.imshow(face_to_plot)
plt.axis("off")
plt.show()

face = face.to(torch.float32) / 255.0

RuntimeError: Numpy is not available

In [16]:
target_layers = [model.block8]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

cam = GradCAM(model=model, target_layers=target_layers)
targets = [ClassifierOutputTarget(0)]

# Run CAM
grayscale_cam = cam(input_tensor=face, targets=targets, eigen_smooth=True)
grayscale_cam = grayscale_cam[0, :]

# Assuming 'face' is a normalized tensor [1, 3, H, W]
face_image_to_plot = face.squeeze(0).permute(1, 2, 0).cpu().numpy()
face_image_to_plot = (face_image_to_plot - face_image_to_plot.min()) / (face_image_to_plot.max() - face_image_to_plot.min())
face_image_to_plot_uint8 = (face_image_to_plot * 255).astype(np.uint8)

# Visualization
visualization = show_cam_on_image(face_image_to_plot, grayscale_cam, use_rgb=True)

# Blend CAM on original face image
face_with_mask = cv2.addWeighted(face_image_to_plot_uint8, 1, visualization, 0.5, 0)

# Show result
plt.imshow(face_with_mask)
plt.axis("off")
plt.show()


NameError: name 'face' is not defined

In [30]:
def predict(input_image:Image.Image, true_label:str):
    """Predict the label of the input_image"""
    face = mtcnn(input_image)
    if face is None:
        raise Exception('No face detected')
    face = face.unsqueeze(0) # add the batch dimension
    face = F.interpolate(face, size=(256, 256), mode='bilinear', align_corners=False)
    
    # convert the face into a numpy array to be able to plot it
    face_image_to_plot = face.squeeze(0).permute(1, 2, 0).cpu().detach().int().numpy()

    face = face.to(DEVICE)
    face = face.to(torch.float32)
    face = face / 255.0
    with torch.no_grad():
        output = torch.sigmoid(model(face).squeeze(0))
        prediction = "real" if output.item() < 0.5 else "fake"
        
        real_prediction = 1 - output.item()
        fake_prediction = output.item()
        
        confidences = {
            'real': real_prediction,
            'fake': fake_prediction
        }
    return confidences, true_label, face_image_to_plot


In [31]:
for i in range(6):
    example = examples[i]
    example_img = example['path']
    example_label = example['label']

    print(f"True label: {example_label}")

    example_img = Image.open(example_img)
    confidences, _, _ = predict(example_img, example_label)
    if confidences['real'] > 0.5:
        print("Predicted label: real")
    else:
        print("Predicted label: fake")

    print()

True label: real
Predicted label: real

True label: fake
Predicted label: fake

True label: real
Predicted label: real

True label: real
Predicted label: real

True label: fake
Predicted label: fake

True label: fake
Predicted label: fake



In [32]:
import gradio as gr

interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(label="Input Image", type="pil"),
        gr.Textbox(label="Some Text")
    ],
    outputs=[
        gr.Label(label="Class"),
        gr.Textbox(label="Text"),
        gr.Image(label="Face", type="pil") 
    ],
)

interface.launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
